# MrVI روی Dataset 2: mBDRC renal cortex با scRareBench (GitHub main)

این نوت‌بوک **هیچ تغییری در core package ایجاد نمی‌کند** و از همان نسخهٔ جاری scRareBench از GitHub استفاده می‌کند.

- Dataset selector: `2` = `mbdrc_renal_cortex`
- benchmark label: `cell_type`
- benchmark batch: `donor_id + assay` (`scrarebench_batch`)
- MrVI sample key: `donor_id`
- MrVI nuisance batch key: `assay`
- benchmark representation: MrVI `u` latent (`give_z=False`)
- rare taxonomy: فقط GR/LE/SR به‌صورت distribution-based؛ DL/RM curate نشده و پنل six-scenario غیرفعال است.


### Dataset 2 preprocessing fix
The executed six-dataset audit showed that global `seurat_v3` HVG selection with `span=0.3` succeeds on mBDRC, while batch-aware HVG with `donor_id+assay` remains LOESS-singular even at larger spans. Therefore **all Dataset 2 method notebooks use the same fixed global 4000-HVG selection from raw counts**. The integration/evaluation metadata are unchanged.


## ۱. نصب مستقیم از GitHub

این notebook دیگر به ZIP محلی نیاز ندارد. ابتدا خود `scrarebench` بدون dependency نصب می‌شود؛ سپس `scrarebench.runtime.setup_notebook()` dependencyهای benchmark و روش انتخاب‌شده را با constraints، `pip check` و smoke-test مدیریت می‌کند.


In [ ]:
SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@main"
METHOD = "mrvi"

# To pin a release/commit, replace @main above with @vX.Y.Z or @<commit-sha>.


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

# Bootstrap only the lightweight package code from GitHub. Dependencies are
# deliberately handled by scrarebench.runtime in the next step.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

from scrarebench.runtime import print_install_report, setup_notebook

install_report = setup_notebook(METHOD, quiet=False)
print_install_report(install_report)

# Import only after dependency validation and the fresh-process smoke test pass.
import scrarebench as _scrarebench_install_check
import scib_metrics as _scib_metrics_install_check
print("scrarebench import path:", Path(_scrarebench_install_check.__file__).resolve())
print("scrarebench version:", _scrarebench_install_check.__version__)
print("scib-metrics version:", getattr(_scib_metrics_install_check, "__version__", "installed"))
import scvi as _method_install_check
from scvi.external import MRVI as _MRVI_install_check
print("scvi-tools version:", _method_install_check.__version__)
print("MRVI import: OK")


## ۲. Imports و GPU


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import hashlib
import importlib
import json
import os
import platform
import shutil
import sys
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import yaml


import scrarebench
from IPython.display import HTML, display

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("scRareBench:", scrarebench.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import scvi
from scvi.external import MRVI
print("scvi-tools:", scvi.__version__)


## ۴. تنظیمات MrVI / Dataset 2


In [ ]:
WORK_DIR = Path("/content/scrarebench_mrvi_dataset2_run_v091_fixed")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = DATA_DIR / "cache"
RESULTS_DIR = WORK_DIR / "results" / "MrVI_dataset2"
ARTIFACT_DIR = WORK_DIR / "deliverable"
for directory in (WORK_DIR, DATA_DIR, CACHE_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATASET_SELECTOR = 2
DATASET_KEY = "mbdrc_renal_cortex"
LABEL_KEY = "cell_type"
BATCH_KEY = "scrarebench_batch"
COUNTS_LAYER = "counts"
LATENT_KEY = "X_mrvi_u"
METHOD_NAME = "MrVI-u"
SEED = 42
N_HVG = 4000

HVG_SELECTION_POLICY = "global_seurat_v3_raw_counts"
HVG_BATCH_KEY = None
HVG_SPAN = 0.3
SCIB_HVG_BATCH_MODE = "global"

N_NEIGHBORS = 15
REFERENCE_RESOLUTION = 1.0
RESOLUTION_SWEEP = (1.0,)
DISTANCE_METRIC = "euclidean"

RUN_SCIB = True
SCIB_N_HVG = 4000
SCIB_REFERENCE_N_PCS = 50
SCIB_N_JOBS = 1
SCIB_PROGRESS_BAR = True
SCIB_INCLUDE_SILHOUETTE_BATCH = True
SCIB_REQUIRE_SUCCESS = True

FORCE_REBUILD_DATASET = False
GENERATE_UMAP = True
GENERATE_INTERACTIVE_REPORT = True
GENERATE_PDF_REPORT = True
DOWNLOAD_RESULT_ZIP = True
DOWNLOAD_STANDALONE_REPORT = False
INCLUDE_LATENT_IN_BUNDLE = True

HTML_INCLUDE_OVERVIEW = True
HTML_INCLUDE_METRICS = True
HTML_INCLUDE_SCIB = True
HTML_INCLUDE_RARE = True
HTML_INCLUDE_RARE_UMAP = True
HTML_INCLUDE_RARE_HEATMAPS = True
HTML_INCLUDE_RARE_SCENARIO_ANALYSIS = False
HTML_INCLUDE_UMAP = True
HTML_INCLUDE_SANKEY = True
HTML_INCLUDE_REPRODUCIBILITY = True
HTML_INCLUDE_STATIC_FIGURES = True
HTML_INCLUDE_CELL_IDS = True

SAMPLE_KEY = 'donor_id'
MRVI_NUISANCE_BATCH_KEY = 'assay'
N_LATENT_U = 30
N_LATENT_Z = 30
ENCODER_N_HIDDEN = 128
ENCODER_N_LAYERS = 2
MAX_EPOCHS = 300
TRAIN_SIZE = 0.90
BATCH_SIZE = 128
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 25
REUSE_MATCHING_MRVI_LATENT = True
MRVI_CACHE_DIR = WORK_DIR / "mrvi_cache"
MRVI_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)
print("Dataset selector:", DATASET_SELECTOR, DATASET_KEY)


## ۵. دانلود Dataset 2، raw counts و rare classes


In [ ]:
def sampled_count_diagnostics(matrix, max_values: int = 100_000, seed: int = 42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    finite = values[np.isfinite(values)]
    return {
        "sampled_values": int(len(values)),
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "maximum": float(np.max(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }

def _copy_matrix(matrix):
    if sp.issparse(matrix):
        return matrix.copy().tocsr()
    return np.asarray(matrix).copy()

def ensure_counts_layer(adata, layer_key="counts"):
    """Create/validate a raw-count layer without changing cell order.

    Priority: existing layer -> CELLxGENE/raw slot aligned to current var_names -> X if X is count-like.
    """
    if layer_key in adata.layers:
        diag = sampled_count_diagnostics(adata.layers[layer_key], seed=SEED)
        if diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(f"Existing {layer_key!r} layer is not nonnegative integer-like raw counts: {diag}")

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)
        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = sampled_count_diagnostics(matrix, seed=SEED)
            if diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = _copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = sampled_count_diagnostics(adata.X, seed=SEED)
    if diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = _copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts. "
        f"layers={list(adata.layers)}, raw_present={adata.raw is not None}, X_diagnostics={diag}"
    )

def gex_mask(adata):
    for key in ("feature_types", "feature_type", "modality"):
        if key not in adata.var.columns:
            continue
        values = adata.var[key].astype(str).str.lower()
        mask = values.str.contains("gene expression|gex|rna", regex=True).to_numpy()
        if mask.any():
            return mask
    if "feature_biotype" in adata.var.columns:
        mask = adata.var["feature_biotype"].astype(str).str.lower().eq("gene").to_numpy()
        if mask.any():
            return mask
    return np.ones(adata.n_vars, dtype=bool)

def build_distribution_only_scenarios(adata, *, batch_key, label_key):
    """Infer GR/LE/SR only from abundance and batch distribution.

    DL/RM topology is deliberately not inferred for external/raw datasets because that
    requires curated biological metadata. This keeps the external benchmark descriptive
    instead of silently imposing the paper-main six-scenario labels.
    """
    from scrarebench.scenarios import infer_distribution_classes, annotate_paper_scenarios

    distribution = infer_distribution_classes(
        adata.obs,
        batch_key=batch_key,
        label_key=label_key,
        global_abundance_threshold=0.01,
        batch_fraction_threshold=0.25,
        local_abundance_threshold=0.01,
    )
    rare = distribution[distribution["distribution"].isin(["GR", "LE", "SR"])].copy()
    if rare.empty:
        raise RuntimeError("No GR/LE/SR rare populations were identified with the configured thresholds.")

    scenario_table = pd.DataFrame({
        "cell_type": rare["cell_type"].astype(str),
        "scenario": rare["distribution"].astype(str),
        "distribution": rare["distribution"].astype(str),
        "topology": "UNASSIGNED",
        "parent_type": "",
        "curation_source": "distribution-only inference (GR/LE/SR); DL/RM not curated",
    })
    annotate_paper_scenarios(
        adata,
        label_key=label_key,
        table=scenario_table,
        prefix="scrarebench_",
        inplace=True,
    )
    return distribution, scenario_table

def hash_strings(values) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\\0")
    return digest.hexdigest()


from scrarebench import load_dataset, resolve_dataset
from scrarebench.scenarios import infer_distribution_classes

spec = resolve_dataset(DATASET_SELECTOR)
assert spec.key == DATASET_KEY, (spec.key, DATASET_KEY)
adata = load_dataset(
    DATASET_SELECTOR,
    DATA_DIR,
    force_download=False,
    force_rebuild=FORCE_REBUILD_DATASET,
    strict_expected_counts=True,
)
if not adata.obs_names.is_unique:
    raise ValueError("Cell barcodes / obs_names are not unique.")

# Dataset 2: mBDRC renal cortex.
# Verified benchmark batch definition: (donor_id, assay); CELLxGENE label: cell_type.
for required_key in ("donor_id", "assay", LABEL_KEY):
    if required_key not in adata.obs.columns:
        raise KeyError(
            f"Dataset 2 requires obs[{required_key!r}]. Available columns: {list(adata.obs.columns)}"
        )
adata.obs[BATCH_KEY] = pd.Categorical(
    adata.obs["donor_id"].astype(str) + " || " + adata.obs["assay"].astype(str)
)


count_source, count_diagnostics = ensure_counts_layer(adata, COUNTS_LAYER)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)

# External/raw datasets receive abundance/distribution-only GR/LE/SR annotation.
distribution_table, scenario_table = build_distribution_only_scenarios(
    adata,
    batch_key=BATCH_KEY,
    label_key=LABEL_KEY,
)

print(adata)
print("Cells:", f"{adata.n_obs:,}")
print("Features:", f"{adata.n_vars:,}")
print("Batches:", adata.obs[BATCH_KEY].astype(str).nunique())
print("Cell types:", adata.obs[LABEL_KEY].astype(str).nunique())
print("Rare distribution counts:")
display(distribution_table["distribution"].value_counts(dropna=False).rename_axis("distribution").to_frame("n_cell_types"))
print("Rare populations used by scRareBench:")
display(scenario_table)

for required_key in [SAMPLE_KEY] + ([MRVI_NUISANCE_BATCH_KEY] if MRVI_NUISANCE_BATCH_KEY else []):
    if required_key not in adata.obs.columns:
        raise KeyError(f"MrVI metadata key {required_key!r} is missing. Available={list(adata.obs.columns)}")
print("MrVI sample key:", SAMPLE_KEY, "n=", adata.obs[SAMPLE_KEY].astype(str).nunique())
print("MrVI nuisance batch key:", MRVI_NUISANCE_BATCH_KEY or "none")


## ۶. HVG / raw-count input برای MrVI


In [ ]:
# Fixed, dataset-level HVG policy. It is identical across methods on the same dataset.
# Dataset 2 audit: global Seurat-v3 HVG at span=0.3 passed, while donor_id+assay batch-aware HVG remained numerically singular. The same fixed global HVG policy is used for scVI, Harmony and MrVI on Dataset 2.
rna_mask = gex_mask(adata)
adata_rna = adata[:, rna_mask].copy() if int(rna_mask.sum()) != adata.n_vars else adata.copy()
adata_rna.var_names_make_unique()

if COUNTS_LAYER not in adata_rna.layers:
    raise KeyError(f"{COUNTS_LAYER!r} layer is missing after count preparation.")

count_diagnostics = sampled_count_diagnostics(adata_rna.layers[COUNTS_LAYER], seed=SEED)
if not count_diagnostics["nonnegative"] or not count_diagnostics["integer_like"]:
    raise ValueError(f"Selected count matrix is invalid: {count_diagnostics}")

hvg_kwargs = {
    "layer": COUNTS_LAYER,
    "flavor": "seurat_v3",
    "n_top_genes": min(N_HVG, adata_rna.n_vars),
    "span": HVG_SPAN,
    "subset": False,
    "check_values": True,
}
if HVG_BATCH_KEY is not None:
    if HVG_BATCH_KEY not in adata_rna.obs.columns:
        raise KeyError(
            f"HVG_BATCH_KEY={HVG_BATCH_KEY!r} is missing from adata.obs. "
            f"Available={list(adata_rna.obs.columns)}"
        )
    hvg_kwargs["batch_key"] = HVG_BATCH_KEY

print("HVG selection policy:", HVG_SELECTION_POLICY)
print("HVG batch key:", HVG_BATCH_KEY if HVG_BATCH_KEY is not None else "<global / no batch_key>")
print("HVG span:", HVG_SPAN)

sc.pp.highly_variable_genes(adata_rna, **hvg_kwargs)

hvg_mask = adata_rna.var["highly_variable"].fillna(False).to_numpy()
n_selected_hvg = int(hvg_mask.sum())
if n_selected_hvg == 0:
    raise RuntimeError("No HVGs were selected.")
if n_selected_hvg != min(N_HVG, adata_rna.n_vars):
    raise RuntimeError(
        f"Expected {min(N_HVG, adata_rna.n_vars)} HVGs but selected {n_selected_hvg}. "
        "Stop here rather than silently changing preprocessing across methods."
    )

hvg_names = adata_rna.var_names[hvg_mask]
counts_hvg = adata_rna.layers[COUNTS_LAYER][:, hvg_mask]
if sp.issparse(counts_hvg):
    counts_hvg = counts_hvg.tocsr().astype(np.float32)
else:
    counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

adata_mrvi = ad.AnnData(
    X=counts_hvg.copy(),
    obs=adata_rna.obs.copy(),
    var=adata_rna.var.loc[hvg_names].copy(),
)
adata_mrvi.var_names_make_unique()

if not np.array_equal(adata_mrvi.obs_names.astype(str), adata.obs_names.astype(str)):
    raise RuntimeError("Cell order changed during method preprocessing.")

print(adata_mrvi)
print("Selected HVGs:", adata_mrvi.n_vars)
print("HVG policy hash input:", HVG_SELECTION_POLICY, HVG_BATCH_KEY, HVG_SPAN)


## ۷. آموزش MrVI یا بازیابی latent cache


In [ ]:
mrvi_config = {
    "seed": SEED, "n_hvg": N_HVG, "n_latent_u": N_LATENT_U, "n_latent_z": N_LATENT_Z,
    "encoder_n_hidden": ENCODER_N_HIDDEN, "encoder_n_layers": ENCODER_N_LAYERS,
    "max_epochs": MAX_EPOCHS, "train_size": TRAIN_SIZE, "batch_size": BATCH_SIZE,
    "sample_key": SAMPLE_KEY, "nuisance_batch_key": MRVI_NUISANCE_BATCH_KEY,
    "dataset_selector": DATASET_SELECTOR, "scvi_tools_version": scvi.__version__,
    "hvg_selection_policy": HVG_SELECTION_POLICY,
    "hvg_batch_key": HVG_BATCH_KEY,
    "hvg_span": HVG_SPAN,
}
manifest = {
    "cell_hash": hash_strings(adata_mrvi.obs_names),
    "gene_hash": hash_strings(adata_mrvi.var_names),
    "sample_hash": hash_strings(adata_mrvi.obs[SAMPLE_KEY].astype(str)),
    "nuisance_hash": hash_strings(adata_mrvi.obs[MRVI_NUISANCE_BATCH_KEY].astype(str)) if MRVI_NUISANCE_BATCH_KEY else None,
    "config": mrvi_config,
}
manifest_path=MRVI_CACHE_DIR/"run_manifest.json"; latent_cache=MRVI_CACHE_DIR/"X_mrvi_u.npy"; history_cache=MRVI_CACHE_DIR/"training_history.csv"
can_reuse=False
if REUSE_MATCHING_MRVI_LATENT and manifest_path.exists() and latent_cache.exists():
    try: can_reuse=json.loads(manifest_path.read_text())==manifest
    except Exception: can_reuse=False
if can_reuse:
    X_latent=np.load(latent_cache).astype(np.float32); training_seconds=0.0
    history_df=pd.read_csv(history_cache) if history_cache.exists() else pd.DataFrame()
    model=None
    print("Loaded matching cached MrVI-u latent.")
else:
    scvi.settings.seed=SEED
    np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    setup_kwargs={"sample_key": SAMPLE_KEY, "backend": "torch"}
    if MRVI_NUISANCE_BATCH_KEY is not None: setup_kwargs["batch_key"] = MRVI_NUISANCE_BATCH_KEY
    MRVI.setup_anndata(adata_mrvi, **setup_kwargs)
    model=MRVI(
        adata_mrvi,
        n_latent=N_LATENT_Z,
        n_latent_u=N_LATENT_U,
        encoder_n_hidden=ENCODER_N_HIDDEN,
        encoder_n_layers=ENCODER_N_LAYERS,
    )
    start=time.perf_counter()
    model.train(
        max_epochs=MAX_EPOCHS, accelerator="auto", devices="auto",
        train_size=TRAIN_SIZE, validation_size=1.0-TRAIN_SIZE,
        batch_size=BATCH_SIZE, early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        check_val_every_n_epoch=1, enable_progress_bar=True,
    )
    training_seconds=time.perf_counter()-start
    X_latent=np.asarray(
        model.get_latent_representation(adata=adata_mrvi, batch_size=BATCH_SIZE, use_mean=True, give_z=False),
        dtype=np.float32,
    )
    history_frames=[]
    for key, values in getattr(model,"history",{}).items():
        frame=pd.DataFrame(values).copy(); frame["metric"]=key; frame["epoch"]=frame.index; history_frames.append(frame.reset_index(drop=True))
    history_df=pd.concat(history_frames,ignore_index=True) if history_frames else pd.DataFrame()
    np.save(latent_cache,X_latent); manifest_path.write_text(json.dumps(manifest,indent=2)); history_df.to_csv(history_cache,index=False)
print("MrVI-u latent:",X_latent.shape)
print(f"Training time: {training_seconds/60:.2f} min")


## ۸. ثبت MrVI-u latent در AnnData اصلی


In [ ]:
from scrarebench.latent import attach_latent
barcodes=adata_mrvi.obs_names.astype(str).to_numpy()
if X_latent.shape != (adata.n_obs, N_LATENT_U): raise RuntimeError(f"Unexpected MrVI-u shape: {X_latent.shape}")
if not np.isfinite(X_latent).all(): raise ValueError("MrVI-u latent contains NaN/Inf")
alignment_report=attach_latent(adata,X_latent,key=LATENT_KEY,latent_barcodes=barcodes,allow_reorder=False,overwrite=True)
np.save(WORK_DIR/f"MrVI_dataset{DATASET_SELECTOR}_u_latent.npy",X_latent); np.save(WORK_DIR/f"MrVI_dataset{DATASET_SELECTOR}_barcodes.npy",barcodes)
print("Alignment:",alignment_report)
if model is not None: del model
del adata_mrvi, counts_hvg
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## ۹. ارزیابی ثابت scRareBench


In [ ]:
from scrarebench.evaluation import EvaluationConfig, evaluate_latent
from scrarebench.scib_backend import ScibEvaluationConfig

if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_config = EvaluationConfig(
    method_name=METHOD_NAME,
    representation_key=LATENT_KEY,
    label_key=LABEL_KEY,
    batch_key=BATCH_KEY,
    scenario_key="scrarebench_scenario",
    reference_resolution=REFERENCE_RESOLUTION,
    resolution_sweep=RESOLUTION_SWEEP,
    n_neighbors=N_NEIGHBORS,
    distance_metric=DISTANCE_METRIC,
    random_state=SEED,
    overwrite=True,
    scib=ScibEvaluationConfig(
        enabled=RUN_SCIB,
        count_layer=COUNTS_LAYER,
        n_hvg=SCIB_N_HVG,
        hvg_batch_mode=SCIB_HVG_BATCH_MODE,
        reference_n_pcs=SCIB_REFERENCE_N_PCS,
        n_jobs=SCIB_N_JOBS,
        progress_bar=SCIB_PROGRESS_BAR,
        include_silhouette_batch=SCIB_INCLUDE_SILHOUETTE_BATCH,
        require_backend=SCIB_REQUIRE_SUCCESS,
    ),
)

# IMPORTANT: raw/external datasets use their own distribution-derived rare table.
# The paper-main six-scenario table is NOT silently reused here.
result = evaluate_latent(
    adata,
    benchmark_config,
    RESULTS_DIR,
    scenario_table=scenario_table,
)

print("Benchmark completed.")
print("Reference cluster key:", result.cluster_keys[REFERENCE_RESOLUTION])
print("\\nOverall / rare / non-rare metrics")
display(result.subset_metrics.round(5))
print("\\nRare-cell summary")
display(result.rare_summary.round(5))
print("\\nRare-cell per-type metrics")
display(result.rare_metrics.round(5))
if result.scib is not None:
    print("\\nscIB-compatible aggregate scores")
    display(result.scib.aggregate_scores.round(5))
    print("\\nscIB-compatible individual metrics")
    display(result.scib.metrics_long.round(5))
    print("\\nMetric applicability/status")
    display(result.scib.metric_status)


## ۱۰. UMAP و گزارش‌ها


In [ ]:
from scrarebench.reporting import write_html_report, write_interactive_report, write_pdf_report

figures_dir = RESULTS_DIR / "rare_cell" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
extra_figures = []
run_config = yaml.safe_load((RESULTS_DIR / "reproducibility" / "run_config.yaml").read_text(encoding="utf-8"))
neighbors_key = run_config["neighbors_key"]
reference_cluster_key = run_config["reference_cluster_key"]

UMAP_KEY = "X_umap_MrVI_dataset2"
if GENERATE_UMAP:
    sc.tl.umap(adata, neighbors_key=neighbors_key, random_state=SEED)
    adata.obsm[UMAP_KEY] = adata.obsm["X_umap"].copy()
    for color_key, title, filename, size in [
        (LABEL_KEY, "MrVI-u latent UMAP — reference cell types", "umap_cell_types.png", (13, 9)),
        (BATCH_KEY, "MrVI-u latent UMAP — benchmark batches", "umap_batches.png", (12, 8)),
    ]:
        sc.pl.embedding(
            adata,
            basis=UMAP_KEY,
            color=color_key,
            title=title,
            frameon=False,
            show=False,
            legend_loc="right margin",
        )
        path = figures_dir / filename
        plt.gcf().set_size_inches(*size)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

    rare_mask = adata.obs["scrarebench_is_rare"].astype(bool).to_numpy()
    if rare_mask.any():
        rare_view = adata[rare_mask].copy()
        sc.pl.embedding(
            rare_view,
            basis=UMAP_KEY,
            color="scrarebench_scenario",
            title="Distribution-defined rare populations — GR / LE / SR",
            frameon=False,
            show=False,
            legend_loc="right margin",
            size=24,
        )
        path = figures_dir / "umap_rare_distribution_classes.png"
        plt.gcf().set_size_inches(11, 8)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

if not history_df.empty:
    fig,ax=plt.subplots(figsize=(9,5.5)); plotted=0
    preferred=[m for m in history_df["metric"].dropna().unique() if "elbo" in str(m).lower()]
    use_metrics=preferred or list(history_df["metric"].dropna().unique())[:4]
    for metric in use_metrics:
        frame=history_df[history_df["metric"]==metric]
        numeric=[c for c in frame.columns if c not in {"metric","epoch"} and pd.api.types.is_numeric_dtype(frame[c])]
        if numeric:
            ax.plot(frame["epoch"],frame[numeric[0]].to_numpy(dtype=float),label=str(metric),alpha=.85); plotted+=1
    if plotted:
        ax.set_xlabel("Epoch"); ax.set_ylabel("Recorded value"); ax.set_title("MrVI training history"); ax.legend(fontsize=7); fig.tight_layout()
        p=figures_dir/"mrvi_training_history.png"; fig.savefig(p,dpi=180,bbox_inches="tight"); plt.close(fig); extra_figures.append(p)
    else: plt.close(fig)


base_figures = []
if result.scib is not None:
    base_figures.append(result.scib.files["metric_plot"])
base_figures.extend([
    result.files["rare_metric_heatmap"],
    result.files["rare_precision_recall"],
    result.files["failure_counts"],
])

report_metadata = {
    "method": METHOD_NAME,
    "dataset_selector": DATASET_SELECTOR,
    "dataset_key": DATASET_KEY,
    "representation_key": LATENT_KEY,
    "n_cells": adata.n_obs,
    "n_dimensions": adata.obsm[LATENT_KEY].shape[1],
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "rare_definition": "GR/LE/SR inferred from abundance and batch distribution; DL/RM topology not assigned",
    "reference_resolution": REFERENCE_RESOLUTION,
    "n_neighbors": N_NEIGHBORS,
    "distance_metric": DISTANCE_METRIC,
    "benchmark_seed": SEED,
    "reference_cluster_key": reference_cluster_key,
    "cell_order_exact_match": alignment_report["exact_order_match"],
    "scib_backend": result.scib.backend if result.scib else "disabled",
    "scib_backend_version": result.scib.backend_version if result.scib else "n/a",
    "MrVI_n_hvg": N_HVG, "MrVI_n_latent_u": N_LATENT_U, "MrVI_n_latent_z": N_LATENT_Z, "MrVI_sample_key": SAMPLE_KEY, "MrVI_nuisance_batch_key": MRVI_NUISANCE_BATCH_KEY or "none", "training_seconds": round(training_seconds,2),
}

report_path = RESULTS_DIR / "report.html"
write_html_report(
    report_path,
    title="scRareBench report — MrVI-u on Dataset 2: mBDRC renal cortex",
    metadata=report_metadata,
    global_table=result.subset_metrics,
    rare_table=result.rare_metrics,
    figure_names=base_figures + extra_figures,
    scib_metrics=result.scib.metrics_long if result.scib else None,
    scib_aggregates=result.scib.aggregate_scores if result.scib else None,
    scib_status=result.scib.metric_status if result.scib else None,
    rare_summary=result.rare_summary,
    scenario_table=result.scenario_metrics,
)

for index, figure_path in enumerate(extra_figures):
    result.files[f"notebook_figure_{index:02d}"] = figure_path

interactive_report_path = RESULTS_DIR / "interactive_report.html"
pdf_report_path = RESULTS_DIR / "summary_report.pdf"
HTML_REPORT_OPTIONS = {
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "scenario_key": "scrarebench_scenario",
    "umap_key": UMAP_KEY if UMAP_KEY in adata.obsm else None,
    "include_overview": HTML_INCLUDE_OVERVIEW,
    "include_metrics": HTML_INCLUDE_METRICS,
    "include_scib": HTML_INCLUDE_SCIB,
    "include_rare": HTML_INCLUDE_RARE,
    "include_rare_umap": HTML_INCLUDE_RARE_UMAP,
    "include_rare_heatmaps": HTML_INCLUDE_RARE_HEATMAPS,
    "include_rare_scenario_analysis": HTML_INCLUDE_RARE_SCENARIO_ANALYSIS,
    "include_umap": HTML_INCLUDE_UMAP,
    "include_sankey": HTML_INCLUDE_SANKEY,
    "include_reproducibility": HTML_INCLUDE_REPRODUCIBILITY,
    "include_static_figures": HTML_INCLUDE_STATIC_FIGURES,
    "include_cell_ids": HTML_INCLUDE_CELL_IDS,
}
if GENERATE_INTERACTIVE_REPORT:
    write_interactive_report(
        adata,
        result,
        interactive_report_path,
        representation_key=LATENT_KEY,
        **HTML_REPORT_OPTIONS,
    )
if GENERATE_PDF_REPORT:
    write_pdf_report(adata, result, pdf_report_path, representation_key=LATENT_KEY)

print("Static HTML report:", report_path)
print("Interactive HTML report:", interactive_report_path if GENERATE_INTERACTIVE_REPORT else "disabled")
print("PDF report:", pdf_report_path if GENERATE_PDF_REPORT else "disabled")


## ۱۱. نمایش گزارش


In [ ]:
display(HTML(report_path.read_text(encoding="utf-8")))

print("\\nGenerated output files:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULTS_DIR))


## ۱۲. بسته‌بندی خروجی


In [ ]:
from scrarebench.reporting import create_report_bundle

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "method": METHOD_NAME,
    "dataset_selector": DATASET_SELECTOR,
    "dataset": "Dataset 2: mBDRC renal cortex",
    "n_cells": int(adata.n_obs),
    "latent_shape": list(adata.obsm[LATENT_KEY].shape),
    "rare_definition": "distribution-only GR/LE/SR; topology UNASSIGNED",
    "include_latent": INCLUDE_LATENT_IN_BUNDLE,
}
(ARTIFACT_DIR / "README_OUTPUT.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

archive_path = create_report_bundle(
    adata,
    result,
    WORK_DIR / "MrVI_dataset2_mBDRC_scRareBench_v0_9_1.zip",
    representation_key=LATENT_KEY,
    include_latent=INCLUDE_LATENT_IN_BUNDLE,
    write_interactive=GENERATE_INTERACTIVE_REPORT,
    write_pdf=GENERATE_PDF_REPORT,
    interactive_report_options=HTML_REPORT_OPTIONS,
)
print("Final ZIP:", archive_path)
print("ZIP size (MB):", round(archive_path.stat().st_size / 1024 / 1024, 2))

if DOWNLOAD_RESULT_ZIP:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception as exc:
        print("Automatic Colab download was not available:", exc)


## نکات

- latent مورد ارزیابی `u` است (`give_z=False`) تا representation اصلی sample-aware correction MrVI برای benchmark استفاده شود.
- `donor_id` sample/target و `assay` nuisance batch در MrVI است؛ evaluation batch برابر ترکیب donor×assay است.
- scRareBench برای این روش تغییر نکرده است؛ فقط notebook method-specific است.


- `HVG_SELECTION_POLICY`, `HVG_BATCH_KEY`, `HVG_SPAN`, and `SCIB_HVG_BATCH_MODE` are written explicitly so the preprocessing decision is reproducible and shared across methods on the same dataset.
- No adaptive per-method fallback is used: if the fixed policy fails, the notebook stops.
